In [ ]:
import os
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from goes2go import GOES

# 1. Rutas de carpetas
BASE_DIR = "/Users/diegoperales/Meteorology/GOES"
RAW_DATA = os.path.join(BASE_DIR, "raw_data")
OUTPUTS = os.path.join(BASE_DIR, "procesados")

# Crear las carpetas si no existen
for carpeta in [RAW_DATA, OUTPUTS]:
    os.makedirs(carpeta, exist_ok=True)

print(f"✅ Entorno listo en: {BASE_DIR}")

In [ ]:
# 1. Definimos la conexión incluyendo la banda desde aquí
# Al poner bands=13 aquí, el objeto 'g' ya sabe qué banda buscar
g = GOES(satellite=19, product="ABI-L2-CMIPF", domain='S', bands=13)

print("Descargando imagen del GOES-19...")

# 2. Ahora, en .latest() ya NO ponemos la banda, solo la carpeta donde guardar
data = g.latest(save_dir=RAW_DATA)

# 3. Ver el resumen
ds = data.ds
print(ds)

In [ ]:
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import numpy as np
import matplotlib.pyplot as plt

# 1. Extraer parámetros de la proyección directamente del archivo
glob_attrs = ds.goes_imager_projection.attrs
height = glob_attrs['perspective_point_height']
lon_0 = glob_attrs['longitude_of_projection_origin']
sweep = glob_attrs['sweep_angle_axis']

# 2. Crear la proyección Geostacionaria manualmente
# Esta es la "forma" en la que el satélite ve la Tierra
dataprojenv = ccrs.Geostationary(central_longitude=lon_0, 
                                 satellite_height=height, 
                                 sweep_axis=sweep)

# 3. Convertir Kelvin a Celsius
temp_celsius = ds.CMI - 273.15

# 4. Crear la figura
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=ccrs.PlateCarree()) # Mapa plano para lat/lon

# 5. Definir el área de Bolivia (Zoom)
ax.set_extent([-71, -57, -23, -9], crs=ccrs.PlateCarree())

# 6. Añadir detalles geográficos
ax.add_feature(cfeature.COASTLINE, linestyle='-', edgecolor='black')
ax.add_feature(cfeature.BORDERS, linestyle='-', edgecolor='yellow', linewidth=1.5)
ax.add_feature(cfeature.STATES, linestyle='--', edgecolor='white', linewidth=0.8)

# 7. Graficar (Usando x e y del satélite para máxima precisión)
im = ax.pcolormesh(ds.x * height, ds.y * height, temp_celsius, 
                   transform=dataprojenv, 
                   cmap='jet_r', vmin=-80, vmax=40)

# 8. Barra de colores
plt.colorbar(im, label='Temperatura de Brillo (°C)', fraction=0.03, pad=0.04)

plt.title(f"GOES-19 - Canal 13 (Infrarrojo)\nBolivia - {ds.t.dt.strftime('%d %B %Y %H:%M UTC').item()}", 
          fontsize=15, fontweight='bold')

plt.show()

In [ ]:
# ==========================================
# 6. GENERACIÓN DEL MAPA
# ==========================================
from matplotlib.colors import LinearSegmentedColormap

# Rango estricto calibrado
VMIN = -90
VMAX = 40
rango_total = VMAX - VMIN 

def pos(temp):
    return (temp - VMIN) / rango_total

# Paleta RAMMB
colors_cira_exact = [
    (pos(-90), '#ffffff'),  # -90°C hacia abajo: Blanco ultra frío
    (pos(-89), '#808080'),  # -89°C: Gris transición tope extremo
    (pos(-80), '#1A1A1A'),  # -80°C: Gris casi negro muy frío
    (pos(-70), '#E50001'),  # -70°C: Rojo vivo
    (pos(-65), '#FE6602'),  # -65°C: Naranja
    (pos(-60), '#FEE500'),  # -60°C: Amarillo brillante
    (pos(-50), '#02FE00'),  # -50°C: Verde brillante
    (pos(-40), '#00157F'),  # -40°C: Azul oscuro
    (pos(-30), '#01EAF3'),  # -30°C: Cian / Celeste CIRA (Entrada brusca al color)
    (pos(-29), '#B8B8B8'),  # -29°C: Gris muy claro (Transición a grises)
    (pos(0),   '#7E7E7E'),  #   0°C: Gris medio
    (pos(10),  '#6C6C6C'),  #  10°C: Gris medio-oscuro
    (pos(20),  '#525252'),  #  20°C: Gris oscuro
    (pos(30),  '#3A3A3A'),  #  30°C: Gris oscuro bajo
    (pos(40),  '#262626')   #  40°C: Negro/Gris tierra caliente
]

# Crear el mapa de colores con tu calibración oficial
cmap_cira_perfecto = LinearSegmentedColormap.from_list('cira_rammb_exact', colors_cira_exact)

fig = plt.figure(figsize=(12, 11), facecolor='white')
ax = plt.axes(projection=ccrs.PlateCarree())

# Zoom a Bolivia (Perfectamente centrado)
ax.set_extent([-70.5, -57.3, -23.2, -9.5], crs=ccrs.PlateCarree())

# --- CAPA 1: LA IMAGEN SATELITAL ---
im = ax.pcolormesh(ds.x * height, ds.y * height, temp_celsius, 
                   transform=dataprojenv, cmap=cmap_cira_perfecto, vmin=VMIN, vmax=VMAX)

# --- CAPA 2: LÍNEAS GEOGRÁFICAS
COLOR_BLANCO = '#ffffff'

ax.add_feature(cfeature.COASTLINE, linestyle='-', edgecolor='#555555', linewidth=0.5)

# Fronteras Internacionales en blanco continuo y estilizado
ax.add_feature(cfeature.BORDERS, linestyle='-', edgecolor=COLOR_BLANCO, linewidth=1.3, zorder=3)

# Departamentos de Bolivia en línea punteada blanca fina pero visible
ax.add_feature(cfeature.STATES, linestyle=(0, (1, 2)), edgecolor=COLOR_BLANCO, linewidth=0.9, zorder=4)

# --- CAPA 3: REJILLA DE LATITUD Y LONGITUD ---
gl = ax.gridlines(draw_labels=True, linestyle=':', edgecolor='#aaaaaa', linewidth=0.5, zorder=5)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10, 'color': 'gray'}
gl.ylabel_style = {'size': 10, 'color': 'gray'}

# --- BARRA DE COLORES HORIZONTAL ---
cbar = plt.colorbar(im, orientation='horizontal', pad=0.06, fraction=0.04, aspect=45)
cbar.set_label('Temperatura de Brillo (°C)', fontsize=11, fontweight='bold', labelpad=8)
cbar.set_ticks([-90, -80, -70, -60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40])
cbar.ax.tick_params(labelsize=10)

# Título formal
fecha_real = ds.t.dt.strftime('%d %B %Y %H:%M UTC').item()
plt.title(f"GOES-19 - Canal 13 (Infrarrojo)\nMonitoreo Bolivia - {fecha_real}", 
          fontsize=14, fontweight='bold', pad=15)

# Guardar con alta resolución (200 DPI) para mantener las líneas blancas nítidas
output_name = f"{OUTPUTS}/GOES19_IR_CIRA_Exacto_Bolivia_{ds.t.dt.strftime('%Y%m%d_%H%M').item()}.png"
plt.savefig(output_name, dpi=200, bbox_inches='tight')
print(f"¡Mapa guardado! {output_name}")

plt.show()